# Chapter 2: Tokens and Embeddings

> **Goal:** Understand how text is turned into numbers (tokenization) and how those numbers carry meaning (embeddings).

---

## 🧠 The Core Problem

Neural networks only understand **numbers** — they cannot process raw text like `"Hello world"`. So before any LLM can work, we need two transformation steps:

```
Step 1: TEXT → TOKEN IDs     (Tokenization)
"Hello world" → [15496, 995]

Step 2: TOKEN IDs → VECTORS  (Embedding lookup)
[15496, 995] → [[0.12, -0.5, 0.3, ...], [0.8, 0.2, -0.1, ...]]
```

These 768-dimensional (or 1024, 4096, etc.) number vectors are what the Transformer actually processes.

---

## 🔡 What Is a Token?

A **token** is NOT always a word. Modern LLMs use **Byte-Pair Encoding (BPE)** which splits text into sub-word pieces:

```
"unhappiness" → ["un", "happ", "iness"]
"ChatGPT"     → ["Chat", "G", "PT"]
"hello"       → ["hello"]  (common words stay whole)
```

**Why sub-words?** Because:
- We can't have a vocabulary with every word in every language (too big)
- We need to handle new words we've never seen
- Sub-words capture meaning ("un" means NOT, "ness" means quality of)

---

## 💻 Exploring Tokenization

In [ ]:
from transformers import AutoTokenizer

# GPT-2's tokenizer uses BPE with 50,257 vocabulary tokens
tokenizer = AutoTokenizer.from_pretrained('gpt2')

def show_tokens(text, tokenizer):
    """Show how a tokenizer splits text and their IDs."""
    token_ids = tokenizer.encode(text)
    tokens = [tokenizer.decode([t]) for t in token_ids]
    
    print(f"Input text : {repr(text)}")
    print(f"Token IDs  : {token_ids}")
    print(f"Tokens     : {tokens}")
    print(f"Token count: {len(token_ids)}")
    print()

# Try different inputs
show_tokens("Hello world!", tokenizer)
show_tokens("Unhappiness is a complex emotion.", tokenizer)
show_tokens("LLMs are transformers trained on text.", tokenizer)
show_tokens("1 + 1 = 2", tokenizer)
show_tokens("def fibonacci(n):", tokenizer)

### Notice:
- Common words like `"Hello"` and `"world"` are single tokens
- Rarer words get split: `"Unhappiness"` → `["Unh", "app", "iness"]`
- Python code has its own tokenization patterns

**Cost implication:** API providers like OpenAI charge per token, not per word. `1 token ≈ 4 characters ≈ 0.75 words` on average.

---

## 🗺️ What Is an Embedding?

An **embedding** is a vector (list of numbers) that represents the **meaning** of a token in a high-dimensional space.

The key property: **similar meanings → similar vectors**.

```
vector("king")   ≈ [0.8, -0.2, 0.5, 0.1, ...]
vector("queen")  ≈ [0.7, -0.3, 0.5, 0.1, ...]  ← similar to king!
vector("potato") ≈ [-0.2, 0.8, -0.3, 0.9, ...]  ← very different
```

The famous analogy:
```
king - man + woman ≈ queen
```
This works because embeddings encode semantic relationships geometrically.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Load a small but powerful sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Test sentences covering different topics
sentences = [
    "The dog barked loudly in the park",
    "A puppy made a noise in the garden",  # Similar to #1
    "I love eating pizza for dinner",
    "My favorite meal is pasta and wine",  # Similar to #3
    "Machine learning models learn from data",
    "Neural networks are trained using gradient descent",  # Similar to #5
    "The stock market fell sharply today",  # Different from all above
]

# Compute embeddings
# Each sentence becomes a 384-dimensional vector
embeddings = model.encode(sentences)

print(f"Embedding shape: {embeddings.shape}")
print(f"  {len(sentences)} sentences × {embeddings.shape[1]} dimensions")

In [ ]:
# Compute the cosine similarity matrix
# Cosine similarity = 1.0 means identical direction, 0.0 means unrelated
sim_matrix = cosine_similarity(embeddings)

# Visualize the similarity matrix as a heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Shorten labels for display
short_labels = [
    "Dog barked", "Puppy noise",
    "Eating pizza", "Meal pasta",
    "ML models", "Neural networks",
    "Stock market"
]

im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha='right')
ax.set_yticklabels(short_labels)

# Annotate with values
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f'{sim_matrix[i][j]:.2f}',
                ha='center', va='center', fontsize=9,
                color='white' if sim_matrix[i][j] > 0.7 else 'black')

ax.set_title('Sentence Similarity Matrix\n(Higher = More Similar)', pad=15)
plt.tight_layout()
plt.show()

print("\n🔍 Key observation:")
print(f"  'Dog barked' vs 'Puppy noise': {sim_matrix[0][1]:.3f} (should be HIGH)")
print(f"  'Dog barked' vs 'Stock market': {sim_matrix[0][6]:.3f} (should be LOW)")

## 🔢 Embeddings in Context vs. Static Embeddings

There are two types of embeddings:

| Type | Example | Key Difference |
|------|---------|----------------|
| **Static** | Word2Vec, GloVe | One fixed vector per word — `"bank"` always has the same vector |
| **Contextual** | BERT, GPT-2 | Vector changes based on surrounding words |

Contextual embeddings are **much better** because they handle ambiguity:
```
"I went to the river bank"   → bank = water's edge (one vector)
"I went to the savings bank" → bank = financial institution (different vector!)
```

In [ ]:
# Demonstrate contextual embeddings with BERT
from transformers import BertTokenizer, BertModel
import torch

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()

def get_word_embedding(sentence, word, tokenizer, model):
    """Extract the contextual embedding of a specific word in a sentence."""
    inputs = tokenizer(sentence, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    with torch.no_grad():
        outputs = model(**inputs)
        # Get the hidden state of the last layer (shape: [1, seq_len, 768])
        hidden_states = outputs.last_hidden_state[0]
    
    # Find the position of our target word
    for i, token in enumerate(tokens):
        if word.lower() in token.lower():
            return hidden_states[i].numpy()
    return None

# Same word "bank" in two very different contexts
sentence_a = "I sat by the river bank and watched the water flow."
sentence_b = "I need to go to the bank to deposit my cheque."

emb_a = get_word_embedding(sentence_a, "bank", bert_tokenizer, bert_model)
emb_b = get_word_embedding(sentence_b, "bank", bert_tokenizer, bert_model)

# Measure the difference
similarity = np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))

print(f"Sentence A: '{sentence_a}'")
print(f"Sentence B: '{sentence_b}'")
print(f"\nSimilarity of 'bank' in both contexts: {similarity:.4f}")
print()
print("If < 1.0, the embeddings are context-dependent (as expected)!")
print("A static embedding would give similarity = 1.0 (always the same vector)")

## ✅ Chapter 2 Summary

| Concept | Key Point |
|---------|----------|
| Tokenization | Splits text into sub-word pieces; ~1 token ≈ 4 chars |
| BPE | Byte-Pair Encoding — a smart way to build a reusable vocabulary |
| Embeddings | Dense vectors that represent meaning in high-dimensional space |
| Cosine similarity | Measures how similar two vectors are (1 = identical direction) |
| Static vs contextual | Contextual embeddings change based on surrounding words |

## 🔮 Up Next: Chapter 3 — Inside the Transformer

Now that we know how text becomes vectors, we explore the **Transformer architecture** — specifically the **Attention mechanism** that revolutionized AI.